# 실험 03: LangSmith (에이전트 관측 및 대규모 평가)

## 1. 실험 제목과 목표
- **제목**: 내 AI 비서의 뇌를 들여다보기
- **목표**: 실무에서 디버깅과 평가 대시보드로 가장 사랑받는 **LangSmith**의 두 가지 핵심 기능(Tracing, Evaluation)을 경험합니다.
  1. **Tracing (관측)**: 에이전트가 어떤 생각으로 무슨 툴을 썼는지 추적하기
  2. **Evaluation (평가)**: 최신 LangSmith 가이드에 따른 Dataset -> Target Function -> Evaluator 파이프라인 구축

## 2. 사전 준비
> **주의**: 이 노트북을 제대로 실행하려면 다음 준비가 필요합니다.
> 1. [LangSmith(smith.langchain.com)](https://smith.langchain.com/) 가입 후 API Key 발급
> 2. 프로젝트 최상단의 `.env` 파일에 발급받은 키와 설정 입력 (완료됨)
> 3. `pip install langsmith` 패키지 설치

## 3. 라이브러리 import

In [9]:
import os
import uuid
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langsmith import Client, evaluate
from langsmith.schemas import Run, Example

## 4. 환경 변수 로드 (LangSmith 연동)
`LANGSMITH_TRACING=true`가 켜져 있으면, 이후 실행되는 LangChain의 모든 동작이 LangSmith 서버로 전송됩니다.

In [10]:
load_dotenv()

print("LangSmith Tracing 설정 확인:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith Project 설정 확인:", os.getenv("LANGSMITH_PROJECT"))

# LangSmith 통신 클라이언트 생성
client = Client()
llm = ChatOpenAI(model="gpt-4o-mini")

LangSmith Tracing 설정 확인: true
LangSmith Project 설정 확인: llm-study-lab


## 5. Tracing (관측) 실험
간단한 번역 체인을 만들고 실행해봅니다. 코드를 실행한 후 LangSmith 웹사이트의 프로젝트 대시보드에 들어가 보세요!

In [11]:
print("=== [Tracing 테스트] ===")
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 번역가입니다. 주어진 문장을 {language}로 번역하세요."),
    ("human", "{text}")
])
chain = prompt | llm

response = chain.invoke({
    "language": "프랑스어",
    "text": "LangSmith를 사용하면 인공지능의 사고 과정을 한눈에 볼 수 있어서 너무 편리합니다!"
})

print("결과:", response.content)
print("\n👉 지금 바로 https://smith.langchain.com 에 접속해서 내가 설정한 프로젝트의 'Traces' 탭을 확인해보세요!")
print("프롬프트에 변수가 어떻게 매핑되었고, LLM이 토큰을 얼마나 썼고, 몇 초가 걸렸는지 모두 시각화되어 있습니다.")

=== [Tracing 테스트] ===
결과: Utiliser LangSmith est très pratique, car cela permet de voir le processus de pensée de l'intelligence artificielle en un coup d'œil !

👉 지금 바로 https://smith.langchain.com 에 접속해서 내가 설정한 프로젝트의 'Traces' 탭을 확인해보세요!
프롬프트에 변수가 어떻게 매핑되었고, LLM이 토큰을 얼마나 썼고, 몇 초가 걸렸는지 모두 시각화되어 있습니다.


## 6. Evaluation (평가) 파이프라인 구축
최신 LangSmith 평가 가이드라인에 따라 **Dataset(데이터셋) -> Target Function(대상 앱) -> Evaluators(평가자)** 3가지 요소를 명확히 분리하여 실행합니다.

In [12]:
print("\n=== [Evaluation 테스트] ===")

# [요소 1] 데이터셋(Dataset) 준비
dataset_name = f"QA_Test_Dataset_{uuid.uuid4().hex[:6]}"
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="프랑스어 번역 퀄리티 평가 데이터셋"
)
client.create_examples(
    inputs=[
        {"text": "안녕하세요", "language": "프랑스어"},
        {"text": "감사합니다", "language": "프랑스어"}
    ],
    outputs=[
        {"expected": "Bonjour"},
        {"expected": "Merci"}
    ],
    dataset_id=dataset.id,
)
print(f"데이터셋 '{dataset_name}' 생성 완료.")

# [요소 2] Target Function (평가 대상이 될 우리 앱/에이전트)
def my_translator_app(inputs: dict) -> dict:
    # 위에서 만든 체인으로 입력값 번역 진행
    res = chain.invoke(inputs)
    return {"result": res.content}

# [요소 3] Evaluators (채점 기준 함수)
def qa_evaluator(run: Run, example: Example) -> dict:
    # run: Target Function이 실행된 결과
    # example: 데이터셋에 들어있는 원본 정답
    pred = run.outputs["result"]  # my_translator_app이 리턴한 값
    expected = example.outputs["expected"] # 데이터셋의 정답
    
    # 정답과 예측이 완전히 같으면 1점(True), 다르면 0점(False)
    score = (expected.lower() in pred.lower()) # 완전 일치 대신 포함 여부로 채점
    return {"key": "exact_match", "score": score}

# 4. 대규모 평가 실험(Experiment) 실행
print("\n대규모 채점을 시작합니다...")
experiment_results = evaluate(
    my_translator_app,            # 1. Target
    data=dataset_name,            # 2. Dataset
    evaluators=[qa_evaluator],    # 3. Evaluators
    experiment_prefix="translator-v2"
)

print("\n✅ 채점 완료! LangSmith 사이트의 'Datasets & Testing' 메뉴에서 방금 만든 데이터셋을 클릭해 결과를 확인하세요!")


=== [Evaluation 테스트] ===
데이터셋 'QA_Test_Dataset_fe7f87' 생성 완료.

대규모 채점을 시작합니다...
View the evaluation results for experiment: 'translator-v2-4b82b9e5' at:
https://smith.langchain.com/o/93c8359f-1fc2-43ae-b038-d83c2ea9468d/datasets/c25bec70-bdd8-4550-901c-057789e77701/compare?selectedSessions=7f746dba-61ba-4638-a934-d1142e8cf417




0it [00:00, ?it/s]


✅ 채점 완료! LangSmith 사이트의 'Datasets & Testing' 메뉴에서 방금 만든 데이터셋을 클릭해 결과를 확인하세요!


## 7. 결과 해석

1. **최신 평가 파이프라인**: 과거에는 평가 모듈이 복잡하게 얽혀있었지만, 최신 LangSmith는 위 코드처럼 `평가할 앱`, `데이터셋`, `채점 함수` 3가지만 명확히 만들어서 `evaluate` 함수에 넣어버리는 직관적인 방식을 채택했습니다.
2. **Experiment 대시보드**: 채점 결과표를 통해 어느 문장에서 번역이 틀렸는지 한눈에 파악할 수 있으며, 이 파이프라인을 구축해 두면 나중에 모델을 GPT-3.5로 낮추거나 프롬프트를 바꿨을 때 성능이 유지되는지(회귀 테스트) 1초 만에 검증할 수 있습니다.